# Experiment 3 — Final independent Test

Đây là bước cuối cùng. Notebook:
1. chọn threshold **chỉ từ OOF Development**;
2. kiểm tra Development/Test không trùng patient;
3. train lại XGBoost trên toàn bộ Development;
4. đánh giá Test **một lần**.

**Không dùng Test để chọn weighting, feature group, hyperparameter hoặc threshold.**

In [23]:
import os, json
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, balanced_accuracy_score
)

RANDOM_STATE = 42
N_SPLITS = 5

# Giữ đúng đường dẫn của notebook hiện tại.
# Nếu dữ liệu thực tế của bạn nằm ở dataset/1219p/... thì chỉ sửa biến này.
DATASET_PATH = "dataset/577p/glcm/kernel5/"
TRAIN_PATH = os.path.join(DATASET_PATH, "dataset.train.csv")
TEST_PATH = "dataset/1200p/glcm/kernel5/dataset.test.csv"

XGB_CONFIG = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    min_child_weight=1,
    gamma=0.0,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

OUTPUT_DIR = "output/577p/glcm/kernel5/final_independent_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Phải đồng bộ với OFFICIAL_OOF_MODE trong notebook 01.
WEIGHTING_MODE = "balanced"  # "balanced" hoặc "none"

# File OOF chính thức do Experiment 1 xuất ra.
OOF_PATH = "output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv"

THRESHOLD_GRID = np.round(np.arange(0.50, 0.951, 0.01), 2)


## 1. Load Development and select GLCM features

In [24]:
train_df = pd.read_csv(TRAIN_PATH)

development = pd.concat([
    train_df.assign(original_split="train")
], ignore_index=True)

PATIENT_CANDIDATES = ["patient_id", "patient", "subject_id", "case_id"]
patient_col = next((c for c in PATIENT_CANDIDATES if c in development.columns), None)

if patient_col is None:
    raise ValueError("Development CSV thiếu patient_id/subject_id/case_id; không được tiếp tục patient-level CV.")
if "label" not in development.columns:
    raise ValueError("Development CSV thiếu cột label.")

development[patient_col] = development[patient_col].astype(str)
train_df[patient_col] = train_df[patient_col].astype(str)

META = {
    "label", "original_split", patient_col,
    "patient_id", "patient", "subject_id", "case_id",
    "x", "y", "z", "i", "j", "k", "voxel_x", "voxel_y", "voxel_z"
}
numeric_cols = [
    c for c in development.columns
    if c not in META and pd.api.types.is_numeric_dtype(development[c])
]
feature_cols = [c for c in numeric_cols]
if not feature_cols:
    raise ValueError("Không tìm thấy cột GLCM trong Development.")

print("Development shape:", development.shape)
print("Development patients:", development[patient_col].nunique())
print("GLCM features:", len(feature_cols))

Development shape: (653397, 27)
Development patients: 577
GLCM features: 24


## 2. Load matching OOF predictions and select threshold on Development only

In [25]:
if WEIGHTING_MODE not in {"balanced", "none"}:
    raise ValueError("WEIGHTING_MODE phải là 'balanced' hoặc 'none'.")

if not os.path.exists(OOF_PATH):
    raise FileNotFoundError(
        f"Không tìm thấy OOF: {OOF_PATH}\n"
        "Hãy chạy notebook 01_XGBoost_GLCM_ClassWeight_Ablation_FIXED.ipynb trước."
    )

oof = pd.read_csv(OOF_PATH)

required_oof = {"label", "prob_tumor"}
if not required_oof.issubset(oof.columns):
    raise ValueError(f"OOF file thiếu cột: {required_oof - set(oof.columns)}")

# Xác nhận OOF đúng weighting đã khóa.
if "condition" in oof.columns:
    conditions_in_oof = set(oof["condition"].dropna().astype(str).unique())
    if conditions_in_oof != {WEIGHTING_MODE}:
        raise RuntimeError(
            f"OOF condition={conditions_in_oof} nhưng WEIGHTING_MODE='{WEIGHTING_MODE}'. "
            "Hãy đồng bộ OFFICIAL_OOF_MODE ở notebook 01 và chạy lại notebook 01."
        )

# Xác nhận OOF bao phủ đúng Development patients.
oof_patient_col = next((c for c in PATIENT_CANDIDATES if c in oof.columns), None)
if oof_patient_col is None:
    raise ValueError("OOF file thiếu patient_id/subject_id/case_id.")

oof[oof_patient_col] = oof[oof_patient_col].astype(str)
if oof_patient_col != patient_col:
    oof = oof.rename(columns={oof_patient_col: patient_col})

dev_ids_for_oof = set(development[patient_col].astype(str).unique())
oof_ids = set(oof[patient_col].astype(str).unique())
if dev_ids_for_oof != oof_ids:
    raise RuntimeError(
        f"OOF/Development patient mismatch: "
        f"missing={len(dev_ids_for_oof - oof_ids)}, extra={len(oof_ids - dev_ids_for_oof)}"
    )

if len(oof) != len(development):
    raise RuntimeError(
        f"OOF rows ({len(oof)}) != Development rows ({len(development)}). "
        "OOF chính thức phải có đúng một prediction cho mỗi Development voxel."
    )

print("Threshold selection OOF:", OOF_PATH)
print("Weighting mode:", WEIGHTING_MODE)
print("OOF rows:", len(oof))
print("OOF patients:", oof[patient_col].nunique())

threshold_rows = []
y_oof = oof["label"].astype(int).to_numpy()
p_oof = oof["prob_tumor"].astype(float).to_numpy()

for threshold in THRESHOLD_GRID:
    pred = (p_oof >= threshold).astype(int)
    threshold_rows.append({
        "threshold": float(threshold),
        "precision": precision_score(y_oof, pred, zero_division=0),
        "recall": recall_score(y_oof, pred, zero_division=0),
        "f1": f1_score(y_oof, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_oof, pred),
        "balanced_accuracy": balanced_accuracy_score(y_oof, pred)
    })

threshold_search = pd.DataFrame(threshold_rows)

# Pre-specified rule: highest F1; tie -> higher Recall; tie -> lower threshold.
threshold_search = threshold_search.sort_values(
    ["f1", "recall", "threshold"],
    ascending=[False, False, True]
).reset_index(drop=True)

LOCKED_THRESHOLD = float(threshold_search.loc[0, "threshold"])
threshold_search.to_csv(
    os.path.join(OUTPUT_DIR, "development_oof_threshold_search.csv"),
    index=False
)

print("LOCKED_THRESHOLD =", LOCKED_THRESHOLD)
display(threshold_search.head(10))


Threshold selection OOF: output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv
Weighting mode: balanced
OOF rows: 653397
OOF patients: 577
LOCKED_THRESHOLD = 0.87


,threshold,precision,recall,f1,mcc,balanced_accuracy
0,0.87,0.714908,0.718637,0.716768,0.697461,0.849577
1,0.88,0.728164,0.705126,0.716460,0.697616,0.843615
2,0.89,0.743430,0.690364,0.715915,0.697906,0.837083
3,0.86,0.701902,0.730201,0.715772,0.696196,0.854559
4,0.90,0.759069,0.674136,0.714086,0.697259,0.829794
5,0.85,0.689147,0.740708,0.713998,0.694320,0.858996
6,0.84,0.676703,0.751070,0.711950,0.692353,0.863337
7,0.91,0.774778,0.655359,0.710082,0.694857,0.821204
8,0.83,0.665965,0.760158,0.709950,0.690570,0.867118
9,0.82,0.655544,0.769414,0.707929,0.688910,0.870964


## 3. Load Test only after configuration and threshold are locked

In [26]:
test_df = pd.read_csv(TEST_PATH)

test_patient_col = next((c for c in PATIENT_CANDIDATES if c in test_df.columns), None)
if test_patient_col is None:
    raise ValueError(
        "Test CSV thiếu patient_id/subject_id/case_id. "
        "Không đánh giá Final Test cho đến khi có thể chứng minh patient disjointness."
    )
if "label" not in test_df.columns:
    raise ValueError("Test CSV thiếu label.")

test_df[test_patient_col] = test_df[test_patient_col].astype(str)

# Normalize patient column name if needed.
if test_patient_col != patient_col:
    test_df = test_df.rename(columns={test_patient_col: patient_col})

dev_ids = set(development[patient_col].unique())
test_ids = set(test_df[patient_col].unique())
overlap = dev_ids & test_ids

print("Development patients:", len(dev_ids))
print("Test patients:", len(test_ids))
print("Development/Test patient overlap:", len(overlap))

if overlap:
    raise RuntimeError(f"STOP: Development/Test patient overlap = {len(overlap)}")

missing_features = [c for c in feature_cols if c not in test_df.columns]
if missing_features:
    raise ValueError(f"Test thiếu {len(missing_features)} GLCM features: {missing_features[:10]}")

print("Test rows:", len(test_df))
print(test_df["label"].value_counts().sort_index())

Development patients: 577
Test patients: 210
Development/Test patient overlap: 0
Test rows: 227195
0    210732
1     16463
Name: label, dtype: int64


## 4. Fit final model on all Development

In [27]:
X_dev = development[feature_cols]
y_dev = development["label"].astype(int)
X_test = test_df[feature_cols]
y_test = test_df["label"].astype(int)

fit_kwargs = {}
class_weight_report = {"weight_0": None, "weight_1": None}

if WEIGHTING_MODE == "balanced":
    classes = np.array([0, 1])
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_dev)
    cw = dict(zip(classes, weights))
    fit_kwargs["sample_weight"] = y_dev.map(cw).to_numpy()
    class_weight_report = {"weight_0": float(cw[0]), "weight_1": float(cw[1])}

final_model = XGBClassifier(**XGB_CONFIG)
final_model.fit(X_dev, y_dev, verbose=False, **fit_kwargs)

print("Final model fitted on all Development.")
print("Weighting:", WEIGHTING_MODE, class_weight_report)

Final model fitted on all Development.
Weighting: balanced {'weight_0': 0.5339929683247712, 'weight_1': 7.854462182045487}


## 5. Final Test — run once

In [28]:
def binary_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
        "roc_auc": roc_auc_score(y_true, prob) if np.unique(y_true).size == 2 else np.nan,
        "pr_auc": average_precision_score(y_true, prob) if np.unique(y_true).size == 2 else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

In [29]:
test_prob = final_model.predict_proba(X_test)[:, 1]
test_metrics = binary_metrics(y_test, test_prob, threshold=LOCKED_THRESHOLD)

final_row = {
    "weighting_mode": WEIGHTING_MODE,
    "locked_threshold": LOCKED_THRESHOLD,
    "development_patients": development[patient_col].nunique(),
    "development_voxels": len(development),
    "test_patients": test_df[patient_col].nunique(),
    "test_voxels": len(test_df),
    "test_non_tumor": int((y_test == 0).sum()),
    "test_tumor": int((y_test == 1).sum()),
    **class_weight_report,
    **test_metrics
}

final_metrics = pd.DataFrame([final_row])
final_metrics.to_csv(os.path.join(OUTPUT_DIR, "final_test_metrics.csv"), index=False)

pred_cols = [patient_col, "label"]
for c in ["x","y","z","i","j","k","voxel_x","voxel_y","voxel_z"]:
    if c in test_df.columns:
        pred_cols.append(c)

test_predictions = test_df[pred_cols].copy()
test_predictions["prob_tumor"] = test_prob
test_predictions["pred_locked_threshold"] = (test_prob >= LOCKED_THRESHOLD).astype(int)
test_predictions.to_csv(os.path.join(OUTPUT_DIR, "final_test_predictions.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "final_protocol.json"), "w", encoding="utf-8") as f:
    json.dump({
        "weighting_mode": WEIGHTING_MODE,
        "locked_threshold": LOCKED_THRESHOLD,
        "xgb_config": XGB_CONFIG,
        "development_test_patient_overlap": len(overlap),
        "threshold_source": "Development OOF only"
    }, f, indent=2)

display(final_metrics)

,weighting_mode,locked_threshold,development_patients,development_voxels,test_patients,test_voxels,test_non_tumor,test_tumor,weight_0,weight_1,...,recall,specificity,f1,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,balanced,0.87,577,653397,210,227195,210732,16463,0.533993,7.854462,...,0.723926,0.957315,0.637735,0.610917,0.931529,0.704493,201737,8995,4545,11918


## Thesis reporting rule

Sau khi cell Final Test này được chạy:
- đây mới là kết quả **Test độc lập cuối cùng**;
- không quay lại thay feature/kernel/weighting/hyperparameter/threshold vì nhìn thấy Test;
- nếu thay bất kỳ thành phần nào, Test không còn là untouched final Test cho cấu hình mới.